In [0]:
%pip install -U \
langchain \
langchain-core \
langchain-community \
langchain-text-splitters \
databricks-langchain \
databricks-vectorsearch \
databricks-agents \
mlflow
# Restart
dbutils.library.restartPython()
# Check Version
import langchain, mlflow
print(f"Langchain Version:{langchain.__version__}", f"MLFlow Version:{mlflow.__version__}")

# Setup and Test LLM and Vector Store connection.

In [0]:
from databricks_langchain import DatabricksVectorSearch
from databricks_langchain.chat_models import ChatDatabricks

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import mlflow
from mlflow.models import infer_signature

# ----------------------------
# CONFIG
# ----------------------------
VECTOR_INDEX = "workspace.default.studebaker_vector_search"
LLM_ENDPOINT = "databricks-llama-4-maverick"

# ----------------------------
# VECTOR SEARCH
# ----------------------------
vector_store = DatabricksVectorSearch(index_name=VECTOR_INDEX)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# ----------------------------
# LLM
# ----------------------------
llm = ChatDatabricks(
    endpoint=LLM_ENDPOINT, temperature=0.1, max_tokens=400
)

# ----------------------------
# PROMPT
# ----------------------------
prompt = ChatPromptTemplate.from_template(
    """You are an expert Studebaker mechanic that responds with repair directions and part numbers for impacted components.
Some pieces of context may be irrelevant, in which case you should not use them to form the answer.
Use the following context to answer the question.
Cite documents when utilized.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Question:
{question}
"""
)


# ----------------------------
# FORMAT DOCUMENTS
# ----------------------------
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# ----------------------------
# LCEL RAG CHAIN
# ----------------------------
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ----------------------------
# RUN QUERY
# ----------------------------
# questions = [
#     "How do I repair the windshield wiper motor on a Studebaker truck?",
#     "How do I replace a front brake shoe on a studebaker truck?"
# ]
# for i, q in enumerate(questions):
#     response = rag_chain.invoke(
#         input=q
#     )
#     print(response, "******\n\n\n")
    # Demo Save off for Ground Truth
    # with open(f"{i}_ground_truth.txt", "w") as file:
    #     file.write(response)

    # With MLFlow 3.x I believe you can generate ground_truth tables from Traces and Spans vs. How I manually do that here.
    # In theory you would be monitoring production, find your golden paths, and generate a DeltaTable and preserve that as your baseline.

# Building and Loading our GroundTruth table for demo purposes.
Because we are using both the Old MLFlow evaluations and the Newer MLFlow evaluations we build a pandas dataframe with the older style.  

I use files because the LLM Outputs are very large, so loading my files into a table below.

In [0]:
# Simple Demo to build a Ground Truth Table in Unity Catalog.
# Loads from a directory of answers, which might be from actual interactions in Production which have been corrected, but might be difficult for the llm to answer.
# Used files because responses can be very long.
# Experiments also has a UI to manually add rows to an evaluation table.

from pyspark.sql import Row

base_directory = "ground_truth"
ground_truths = [
    {
        "question": "How do I repair the windshield wiper motor on a Studebaker truck?",
        "answer_file_path": f"{base_directory}/windshield_repair_ground_truth.txt",
        "expected_facts": [
            "Remove the four retaining bolts and remove the motor (3, Fig. 35)",
            "Remove the cover (5, Fig. 35) and the main link nut (6) to remove the main link (4) from the wiper motor."
            ]
    },
    {
        "question": "How do I replace a front brake shoe on a studebaker truck?",
        "answer_file_path": f"{base_directory}/brake_repair_ground_truth.txt",
        "expected_facts": [
            "Place a wheel cylinder clamp across the boots of the wheel cylinder (3, Fig. 20) to prevent the pistons from coming out.",
            "Install the plain washers (2) and C washers (1)."
        ]
    }
]

def build_groundtruth():
    eval_data = []
    for gt in ground_truths:
        with open(gt["answer_file_path"]) as f:
            answer = f.read()
            eval_data.append(Row(question=gt["question"], answer=answer, expected_facts=gt["expected_facts"]))
    df = spark.createDataFrame(eval_data)
    df.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable("studebaker_groundtruth")
    return df

groundtruth_df = build_groundtruth()
display(groundtruth_df)

# Create the Evaluations Dataframe

I create the evaluations dataframe with the older style (2.x) of MLFlow. This notebook spans both versions.

In [0]:
import time
import pandas as pd


def load_groundtruth():
    return spark.table("studebaker_groundtruth").toPandas()

# Load Fresh GroundTruth
groundtruth_df = load_groundtruth()


# Generate Predictions, Add to Dataframe
def generate_eval_df(df):

    rows = []

    for _, row in df.iterrows():
        # Get Ground Truth
        question = row["question"]
        answer = row["answer"]
        expected_facts = row["expected_facts"]

        # Run Rag Retriever and Log Results
        docs = retriever.invoke(question)
        retrieved_docs = [d.page_content for d in docs]
        context = "\n\n".join(retrieved_docs)
        
        # Calculate Latency
        start = time.time()
        response = rag_chain.invoke(input=question)
        latency = time.time() - start
        
        # Create Eval Row
        rows.append(
            {
                "question": question,
                "answer": answer,
                "prediction": response,
                "context": context,
                "retrieved_docs": retrieved_docs,
                "expected_facts": expected_facts,
                "latency": latency
            }
        )

    return pd.DataFrame(rows)

# Should include (question | answer | prediction | contexts | retrieved_docs | expected_facts | latency)
eval_df = generate_eval_df(groundtruth_df)
display(eval_df)

# MLFlow Evaluations >= 3.0.0 Example

This is the latest MLFlow and utilizes an Expectations Framework where you are generating live predictions during the MLFlow Execution

Full list of Judges (Scorers) -> https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/predefined/#available-judges  
Eval Examples -> https://mlflow.org/docs/latest/genai/eval-monitor/running-evaluation/eval-examples/

Retrieval Judges requires traces.

Some Scores also require GroundTruth or expected_facts.

In [0]:
# MLFLOW 3.X Rag Pipeline
import mlflow
import pandas as pd
from mlflow.genai import evaluate
from mlflow.genai.scorers import (
    Correctness,
    RelevanceToQuery,
    Equivalence,
    RetrievalGroundedness,
    RetrievalRelevance
)

if mlflow.active_run():
    mlflow.end_run()

# Enable LangChain auto-tracing so MLflow captures retriever spans
mlflow.langchain.autolog()

# Define predict_fn — MLflow passes inputs dict keys as kwargs.
# Wrapping rag_chain so MLflow captures retriever traces automatically.
@mlflow.trace
def predict_fn(question):
    return rag_chain.invoke(question)

# Data: inputs + expectations only (no outputs — predict_fn generates them live)
# Utilizing eval_df from above, only using the new MLFLOW Format here.
eval_data_3x = pd.DataFrame([
    {
        "inputs": {"question": row["question"]},
        "expectations": {
            "expected_response": row["answer"]
        }
    }
    for _, row in eval_df.iterrows()
])
print(eval_data_3x)
with mlflow.start_run(run_name="rag_3x_eval"):

    results = evaluate(
        data=eval_data_3x,
        predict_fn=predict_fn,
        scorers=[
            Correctness(),
            RetrievalGroundedness(),
            RetrievalRelevance(),
            RelevanceToQuery(),
            Equivalence()
        ]
    )

print(results.metrics)
# Convert complex object columns to strings for display compatibility
eval_results_display = results.tables["eval_results"].copy()
for col in eval_results_display.columns:
    if eval_results_display[col].dtype == "object":
        eval_results_display[col] = eval_results_display[col].astype(str)
display(eval_results_display)

# MLFlow 2.x to 3.x (older style)

Since this notebook spans the older and now newer style of MlFlow we preserved this style below.

In [0]:
# import pandas as pd
# import mlflow
# from mlflow.deployments import set_deployments_target
# from mlflow.metrics import precision_at_k, recall_at_k, ndcg_at_k, latency
# from mlflow.metrics.genai import relevance, answer_correctness, answer_relevance, answer_similarity, faithfulness
# import time

# if mlflow.active_run():
#     mlflow.end_run()

# # Define LLM as Judge Evaluation
# set_deployments_target("databricks")
# relevance_metric = relevance(
#     model="endpoints:/databricks-llama-4-maverick"
# )

# faithfulness_metric = faithfulness(
#     model="endpoints:/databricks-llama-4-maverick"
# )

# answer_correctness = answer_correctness(
#     model="endpoints:/databricks-llama-4-maverick"
# )

# # ----------------------------
# # START MLflow RUN
# # ----------------------------
# with mlflow.start_run(run_name="studebaker_rag_chain") as run:

#     # Log Evaluation data to mlflow, can use later
#     mlflow.log_table(
#         eval_df,
#         artifact_file="rag_eval_dataset.json"
#     )
#     # Log chain config
#     mlflow.log_param("vector_index", VECTOR_INDEX)
#     mlflow.log_param("llm_endpoint", LLM_ENDPOINT)
#     mlflow.log_param("retriever_k", 5)
#     mlflow.log_param("temperature", 0.1)

#     # Start Evaluations for relevance and latency
#     # Other Options text-generation, classification, regression
#     results = mlflow.evaluate(
#         data=eval_df,
#         predictions="prediction",
#         targets="answer",
#         evaluator_config={
#             "col_mapping":{
#                 "inputs": "question",
#                 "context": "context"
#             }
#         },
#         evaluators="default",
#         extra_metrics = [
#             relevance_metric,
#             faithfulness_metric,
#             answer_correctness,
#             precision_at_k(5),
#             recall_at_k(5),
#             ndcg_at_k(5),
#             latency()
#         ]
#     )

#     print(results.metrics)


#     print(f"MLflow run completed. Run ID: {run.info.run_id}")


# display(results.tables["eval_results_table"])